# 🚀 [ICML 2026] LiDAR: Lookahead Sample Reward Guidance on Kaggle
### Tái lập Thực nghiệm: `LiDAR (DPM-5 / n=50)` trên GenEval Benchmark

**Bài báo:** *Lookahead Sample Reward Guidance for Test-Time Scaling of Diffusion Models* ([arXiv:2602.03211](https://arxiv.org/abs/2602.03211))  
**GitHub Repository:** [github.com/leekwanreal/Noisy-Reward](https://github.com/leekwanreal/Noisy-Reward)  

**Lưu ý quan trọng trên Kaggle:**
- Ở thanh menu bên phải **(Notebook Options)**: Bật **`Accelerator: GPU T4 x2`** (hoặc `GPU P100`) và **`Internet: On`**.

## 📦 Step 1: Kiểm tra GPU & Cài đặt Môi trường Chuẩn

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

# 2. Tải mã nguồn Noisy-Reward về Kaggle
import os
%cd /kaggle/working
if not os.path.exists("/kaggle/working/Noisy-Reward"):
    !git clone https://github.com/leekwanreal/Noisy-Reward.git
%cd /kaggle/working/Noisy-Reward
!git pull origin main

# 3. Cài đặt các thư viện cần thiết
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas

# 4. Tải file vocab cho hpsv2
import urllib.request, hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)

print("\n✅ Môi trường trên Kaggle đã được cài đặt hoàn tất!")

## ⚡ Step 2: Phase 1 — Lookahead Sampling (DPM-5, n=50 particles)
- Sinh 50 hạt qua 5 bước DPMSolver cho mỗi prompt.
- Tối ưu tốc độ cao: Chỉ lưu vector `latent.pt` và điểm số `results.json` vào ổ cứng SSD cục bộ của Kaggle (tốc độ siêu nhanh ~25s / prompt).

In [ ]:
%cd /kaggle/working/Noisy-Reward

!python lookahead_sampling.py \
    --seed=100 \
    --model_name="runwayml/stable-diffusion-v1-5" \
    --prompt_path="prompt_files/geneval_metadata.jsonl" \
    --output_dir="/kaggle/working/LiDAR_Experiment/Lookahead_samples" \
    --num_particles=50 \
    --num_inference_steps=5 \
    --metrics_to_compute="ImageReward"

## 🎯 Step 3: Phase 2 — LiDAR Target Sampling (DDIM 50-steps, 4 ảnh / prompt)
- Sử dụng 50 hạt Lookahead làm mồi dẫn đường LiDAR ($w=12.5$).
- Sinh 4 bức ảnh chất lượng cao hoàn thiện cho mỗi prompt và lưu vào `/kaggle/working/LiDAR_Experiment/Target_samples`.

In [ ]:
%cd /kaggle/working/Noisy-Reward

!python LiDAR_sampling.py \
    --seed=100 \
    --use_rag \
    --model_name="runwayml/stable-diffusion-v1-5" \
    --prompt_path="prompt_files/geneval_metadata.jsonl" \
    --output_dir="/kaggle/working/LiDAR_Experiment/Target_samples" \
    --num_inference_steps=50 \
    --num_particles=4 \
    --top_k=50 \
    --scale=12.5 \
    --resample_t_end=200 \
    --lookahead_path="/kaggle/working/LiDAR_Experiment/Lookahead_samples/100_50_5" \
    --metrics_to_compute="ImageReward" \
    --save_individual_images

## 📊 Step 4: Đánh giá Toàn diện (ImageReward, CLIP, HPS v2.1) & Đối chứng Bảng 2

In [ ]:
import os, glob, json, gc, torch
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# 1. Tìm thư mục kết quả mới nhất
target_runs = sorted(glob.glob("/kaggle/working/LiDAR_Experiment/Target_samples/*"))
if not target_runs:
    raise FileNotFoundError("Chưa tìm thấy thư mục kết quả. Hãy đảm bảo Step 3 đã chạy xong!")

latest_dir = target_runs[-1]
print(f"📂 Đang phân tích kết quả tại: {latest_dir}")

# 2. Thu thập danh sách ảnh và prompt
all_images = []
all_prompts = []
prompt_dirs = sorted(glob.glob(os.path.join(latest_dir, "[0-9]*")))

for p_dir in prompt_dirs:
    meta_path = os.path.join(p_dir, "metadata.jsonl")
    if os.path.exists(meta_path):
        with open(meta_path, "r") as f:
            data = json.load(f)
            prompt_text = data.get("prompt", "")
    else:
        prompt_text = ""
    for img_path in sorted(glob.glob(os.path.join(p_dir, "*.png"))):
        if "grid" not in img_path:
            all_images.append(img_path)
            all_prompts.append(prompt_text)

print(f"🖼️ Tổng số ảnh sinh ra: {len(all_images)} ảnh trên {len(prompt_dirs)} prompts.")

# 3. Đọc ImageReward đã lưu
metrics_file = os.path.join(latest_dir, "final_metrics.json")
ir_mean = 0.0
if os.path.exists(metrics_file):
    with open(metrics_file, "r") as f:
        m = json.load(f)
        ir_mean = m.get("ImageReward", {}).get("mean", 0.0)

# 4. Tính CLIP Score tuần tự
print("\n⏳ Đang tính CLIP-Score...")
%cd /kaggle/working/Noisy-Reward
from fkd_diffusers.rewards import do_clip_score
clip_scores = []
for idx in tqdm(range(0, len(all_images), 10)):
    batch_imgs = [Image.open(p) for p in all_images[idx:idx+10]]
    batch_prompts = all_prompts[idx:idx+10]
    scores = do_clip_score(images=batch_imgs, prompts=batch_prompts)
    clip_scores.extend(scores)
clip_mean = sum(clip_scores) / max(1, len(clip_scores))

# Giải phóng bộ nhớ
gc.collect()
torch.cuda.empty_cache()

# 5. In bảng đối chứng Bảng 2
print("\n" + "="*75)
print("📈 KẾT QUẢ ĐỐI CHỨNG THỰC NGHIỆM VS BÀI BÁO (TABLE 2 - SD v1.5 LiDAR DPM-5/n=50)")
print("="*75)
print(f"• ImageReward (IR):        {ir_mean:.4f}  | Bài báo Table 2: 0.378 ~ 0.384")
print(f"• CLIP Score:             {clip_mean:.4f}  | Bài báo Table 2: 0.278")
print("="*75)

# 6. Hiển thị ảnh mẫu
sample_grid = os.path.join(latest_dir, "00000/grid.png")
if os.path.exists(sample_grid):
    plt.figure(figsize=(16, 5))
    plt.imshow(Image.open(sample_grid))
    plt.axis("off")
    plt.title("4 Particles Generated with LiDAR (Sorted by Reward)", fontsize=14)
    plt.show()

## 💾 Step 5: Nén & Tải Toàn bộ Kết quả về Máy tính
Nén toàn bộ ảnh đẹp của Pha 2 thành file `.zip` để tải về máy từ mục **Output (bên phải)**.

In [ ]:
!zip -r -q /kaggle/working/LiDAR_Target_Results.zip /kaggle/working/LiDAR_Experiment/Target_samples
print("\n✅ Đã nén xong toàn bộ kết quả thành công: /kaggle/working/LiDAR_Target_Results.zip")
print("📥 Bạn có thể tải file ZIP về máy tính tại mục 'Output' ở cột bên phải giao diện Kaggle!")

## 🧪 (Tùy chọn) Chạy Bộ 3 Bài Test Lipschitz & Phân tích Đột phá
Chạy đo đạc độc lập 3 bài test lý thuyết để vẽ biểu đồ so sánh giữa LiDAR gốc vs Phương pháp của bạn.

In [ ]:
%cd /kaggle/working/Noisy-Reward

!python test_lidar_vs_smoothed_surrogate.py \
    --num_prompts=-1 \
    --num_particles=20 \
    --sigma=0.05 \
    --lookahead_dir="/kaggle/working/LiDAR_Experiment/Lookahead_samples/100_50_5" \
    --output_dir="/kaggle/working/test_results"